# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/martindiarua/ML_01/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I use the numerical features identified in ML-04 as the initial feature vector. They capture search demand, competition, content characteristics, freshness, engagement, traffic, and recent trends.

The selected features are all numerical, so no categorical encoding is required at this stage. Missing numerical values are filled using the median value of each feature. This creates a consistent feature frame that can be used for the initial ML experiments.

In [ ]:
!git clone https://github.com/martindiarua/ML_01.git
%cd ML_01/data/raw

import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")


Cloning into 'ML_01'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 154 (delta 60), reused 93 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 1.86 MiB | 14.78 MiB/s, done.
Resolving deltas: 100% (60/60), done.
/content/ML_01/data/raw/ML_01/data/raw/ML_01/data/raw


In [ ]:
# Features selected in ML-04
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "content_age_days",
    "ctr",
    "avg_position",
    "engagement_rate",
    "trend_pct",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d"
]

# Build the feature frame
X = df[features].copy()

# Fill missing numerical values with the median
X = X.fillna(X.median(numeric_only=True))

print("Feature vector shape:", X.shape)

display(X.head())

print("\nMissing values after filling:")
print(X.isna().sum())

Feature vector shape: (30000, 13)


,search_volume,competition,cpc,word_count,content_age_days,ctr,avg_position,engagement_rate,trend_pct,days_since_last_update,impressions_90d,clicks_90d,sessions_90d
0,10.0,0.67,2.05,3221.0,187,0.76,10.6,5.88,-41.4,20,3803,29,17
1,90.0,0.01,0.05,2481.0,445,0.05,20.3,0.00,-57.7,25,15320,7,9
2,0.0,0.00,0.00,3515.0,141,0.09,36.5,0.00,-60.9,20,12581,11,11
3,10.0,0.00,0.00,2877.0,463,0.49,6.2,1.28,-13.8,22,11751,58,78
4,0.0,0.00,0.00,2803.0,263,0.13,44.0,0.00,-34.7,14,19140,24,145



Missing values after filling:
search_volume             0
competition               0
cpc                       0
word_count                0
content_age_days          0
ctr                       0
avg_position              0
engagement_rate           0
trend_pct                 0
days_since_last_update    0
impressions_90d           0
clicks_90d                0
sessions_90d              0
dtype: int64


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## Feature notes

- **search_volume:** Estimated search demand for the topic. Missing values are filled with the median. Available before the refresh decision.
- **competition:** Search competition for the topic. Missing values are filled with the median. Available before the decision.
- **cpc:** Cost-per-click indicator for the search term. Missing values are filled with the median. Available before the decision.
- **word_count:** Number of words in the page content. Missing values are filled with the median. Available before the decision.
- **content_age_days:** Age of the content in days. Available before the decision.
- **ctr:** Click-through rate observed during the reporting period. Available at the decision moment if the decision is based on that reporting period.
- **avg_position:** Average search position during the reporting period. Available at the decision moment.
- **engagement_rate:** Observed engagement rate during the reporting period. Available at the decision moment.
- **trend_pct:** Change in performance over the available historical period. Missing values are filled with the median. Available at the decision moment.
- **days_since_last_update:** Number of days since the page was last updated. Available before the decision.
- **impressions_90d:** Search impressions over the previous 90 days. Available at the decision moment.
- **clicks_90d:** Search clicks over the previous 90 days. Available at the decision moment.
- **sessions_90d:** Sessions over the previous 90 days. Available at the decision moment.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

## 3. The leakage hunt

I tested the feature set for data leakage by deliberately creating one feature, `leak_feature`, that is copied directly from the proxy label `refresh_priority`. This feature would not be available at the decision moment in a real prediction system because it is the answer the model is supposed to predict.

I first trained a model using the selected features without the leaked column and recorded the honest accuracy. I then added `leak_feature` and trained the same type of model again. The accuracy with the leaked feature should become artificially high because the model has been given the label directly.

This demonstrates why label-derived information must never be included as an input feature. After the test, I removed `leak_feature` so that it cannot be used in the final feature set.

I also treat future or post-decision information as a potential leakage risk and exclude any field that would only become known after the refresh decision.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Create the proxy label for the leakage demonstration
df["refresh_priority"] = (
    (df["ctr"] < 0.05) &
    (df["engagement_rate"] < 0.40)
).astype(int)

# Create a deliberately leaked feature
df["leak_feature"] = df["refresh_priority"]

# Build model data
model_df = df[features + ["refresh_priority", "leak_feature"]].dropna()

X = model_df[features]
y = model_df["refresh_priority"]

# Honest model
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

honest_pred = model.predict(X_test)

print("Honest Accuracy:", accuracy_score(y_test, honest_pred))

# Model with deliberate leakage
X_leaky = model_df[features + ["leak_feature"]]

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky, y, test_size=0.2, random_state=42
)

leaky_model = DecisionTreeClassifier(random_state=42)
leaky_model.fit(X_train, y_train)

leaky_pred = leaky_model.predict(X_test)

print("Accuracy WITH leakage:", accuracy_score(y_test, leaky_pred))

# Remove leaked feature
model_df.drop(columns=["leak_feature"], inplace=True)

print("Leak feature removed. The final feature set does not contain the label-derived column.")

Honest Accuracy: 0.9997224535109631
Accuracy WITH leakage: 1.0
Leak feature removed. The final feature set does not contain the label-derived column.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## 4. What I excluded and why

- **content_id:** Identifier only; it provides no meaningful predictive information.
- **client_id:** Identifier for the client; excluded to avoid learning client-specific patterns rather than general content patterns.
- **content_type:** Categorical context field; excluded from this first numerical baseline because categorical encoding has not been introduced yet.
- **main_intent:** Describes search intent; excluded from the initial baseline because it requires categorical handling.
- **provider_used:** Describes the provider used; excluded because it is not a direct measure of the content's refresh need.
- **model_used:** Describes the model used to produce content; excluded because it may capture production-process information rather than the page's refresh need.
- **trend_direction:** Categorical version of trend; excluded because `trend_pct` already provides a numerical representation of the trend.
- **age_tier:** Categorical version of content age; excluded because `content_age_days` provides the underlying numerical measure.
- **freshness_tier:** Categorical version of freshness; excluded because `days_since_last_update` provides a numerical measure.
- **word_count_tier:** Categorical version of word count; excluded because `word_count` provides the numerical measure.
- **char_count and char_count_tier:** Excluded from the initial feature vector because they duplicate information about content length already represented by `word_count`.
- **impressions_last_30d, clicks_last_30d, sessions_last_30d:** Excluded from this first baseline because they overlap with the broader 90-day traffic measures.
- **impressions_prev_30d, clicks_prev_30d, sessions_prev_30d:** Excluded from the initial baseline to keep the feature set small while avoiding unnecessary overlapping traffic measures.
- **scroll_events_90d, scroll_rate, ai_sessions_90d, ai_traffic_pct, pageviews_90d, users_90d, days_with_impressions, days_with_sessions:** Excluded from the initial baseline because they add additional engagement or traffic signals that are not necessary for the first feature vector.
- **refresh_priority:** This is the proxy label, not an input feature. It must not be included as a normal feature.
- **leak_feature:** Deliberately created for the leakage experiment only. It is removed before the final feature set is used.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✔️] Every section above is filled — markdown thinking AND the code that backs it
- [✔️] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔️] No client names, URLs, or private queries anywhere
- [✔️] My claims use careful words: observed, measured, directional, decision-support
- [✔️] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.